# CMIP – Exploratory Data Analysis (EDA)

EC-Earth3 wave and wind data from Francesco Niosi (MOREnergy Lab, Politecnico di Torino): one MATLAB file per scenario.

### Questions
1. Which model and scenario does each file represent?
2. What variables, units, and dimensions are available?
3. Which years and time intervals are covered?
4. Where are values missing?
5. Do the datasets use comparable grids and variable definitions?

Plus Francesco's suggestion: compare **mean, standard deviation and 99th percentile of Hs** over a 10-year period for all three scenarios, as maps.

> **Data status:** the files are still downloading. `Scenario` reads the partial downloads (complete for ~21 years of historical and ~14–15 years of each SSP) and switches to the complete files automatically once they exist. Statistics for the first 10 years are cached in `data/processed/stats/`; those years are already fully downloaded, so the cached results stay valid.

## 0. Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from cmip_utils import (Scenario, SCENARIOS, VARIABLES, EXPECTED_SIZES, STEPS_PER_DAY, RAW,
                        save_fig, save_table, inspect_mat, assumed_times,
                        load_grid, nearest_point, to_map, period_stats, plot_map, area_mean)

warnings.filterwarnings("ignore", message="Mean of empty slice")              # sea-ice points are all NaN
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
pd.set_option("display.float_format", "{:,.3f}".format)
FIG_DIR = "eda"            # figures go to fig/eda/, tables to data/processed/eda/

grid = load_grid()         # coordinates of all sea points (checks the sea mask)

## 1. Which model and scenario does each file represent?

The share contains one file per scenario: `EC/Mat_Data/<scenario>/EC-EARTH3.mat`, plus the sea mask `SeaMask_WW3.mat` and Francesco's MATLAB script.

- **Model:** EC-Earth3 (CMIP6). The sea mask name and the variable names (`hs`, `fp`, `uwnd`, `vwnd`) are those of the WAVEWATCH III wave model, so the wave fields were most likely computed with WW3 driven by EC-Earth3 winds *(to confirm with Francesco)*.
- **Scenarios:** *historical* (simulation of the past climate), *SSP1-2.6* (low emissions, strong mitigation) and *SSP5-8.5* (very high emissions).

In [ ]:
DESCRIPTION = {"historical": "historical climate",
               "ssp126": "SSP1-2.6: low emissions",
               "ssp585": "SSP5-8.5: very high emissions"}
rows = []
for s in SCENARIOS:
    f = RAW / s / "EC-EARTH3.mat"
    size = f.stat().st_size if f.exists() else f.with_name(f.name + ".part").stat().st_size
    with Scenario(s) as sc:
        n = sc.available_steps()
        rows.append({"scenario": s, "description": DESCRIPTION[s],
                     "model": "EC-Earth3 (CMIP6), wave model WW3 (inferred)",
                     "downloaded [GiB]": size / 2**30,
                     "complete [%]": 100 * size / EXPECTED_SIZES[s],
                     "readable time steps": n,
                     "readable years (if 3-hourly)": n / STEPS_PER_DAY / 365.25})
files = pd.DataFrame(rows).set_index("scenario")
display(files)
save_table(files, "files_overview", FIG_DIR)

The sea-mask file also stores a few descriptive fields, and Francesco's script shows how he reads the data:

In [ ]:
import h5py
with h5py.File(RAW / "SeaMask_WW3.mat", "r") as f:
    for k in ("NcFile", "VarName"):                       # MATLAB text is stored as character codes
        print(f"{k:16s}", "".join(map(chr, np.asarray(f[k][()]).ravel())).strip())
    for k in ("Nsea", "LAT", "LON", "UseAllTimeSteps"):
        print(f"{k:16s}", np.asarray(f[k][()]).ravel())

In [ ]:
script = next(RAW.glob("Load_*grid_point.m"), None)
print(script.read_text(errors="replace") if script else "MATLAB script not found in data/raw/")

## 2. What variables, units and dimensions are available?

| Variable | Meaning | Unit | Dimensions (as read in Python) |
|---|---|---|---|
| `hs` | significant wave height | m | time × sea point |
| `fp` | peak frequency (peak period Tp = 1/fp) | Hz | time × sea point |
| `uwnd` | eastward wind component (assumed) | m/s | time × sea point |
| `vwnd` | northward wind component (assumed) | m/s | time × sea point |

The `.mat` files store no units; they are inferred from the WW3 variable names and checked against the value ranges below. Each variable is single precision (float32), stored in chunks of 248 time steps × 66 points with gzip compression. Each file also contains a `metadata` struct, which is at the end of the file and not readable until the download is complete.

In [ ]:
# Value ranges over the first 31 days, all sea points (~140 MB per read, about a minute in total)
rows = []
for s in SCENARIOS:
    with Scenario(s) as sc:
        for v, (meaning, unit) in VARIABLES.items():
            x = sc.read(v, slice(None), slice(0, 248))
            finite = x[np.isfinite(x)]
            p1, p50, p99 = np.percentile(finite, [1, 50, 99])
            rows.append({"scenario": s, "variable": v, "unit": unit,
                         "min": finite.min(), "p1": p1, "median": p50, "p99": p99,
                         "max": finite.max(), "missing [%]": 100 * (1 - finite.size / x.size)})
values = pd.DataFrame(rows).set_index(["variable", "scenario"]).sort_index()
display(values)
save_table(values, "value_ranges_first_31_days", FIG_DIR)

**What to check:** `hs` should be a few metres (rarely above ~15 m), `fp` mostly 0.05–0.3 Hz, and the winds signed values of a few to ~30 m/s. Similar ranges across the three scenarios also show that the files use the same variable definitions (question 5).

## 3. Which years and time intervals are covered?

Each file has **87,656 time steps**. That is exactly 30 years of **3-hourly** data with 7 leap days (10,957 days × 8 steps per day), and each storage chunk holds 248 steps = 31 days.

- **historical:** most likely **1985–2014**, the last 30 years of the CMIP6 historical experiment.
- **SSP files:** also 30 years, but the period is not stored in the readable part of the file. Windows such as 2071–2100 have exactly this length. *To confirm with Francesco.*

Two checks from the data itself: the time series below, and the seasonal cycle. If each file starts on 1 January, the northern-hemisphere wave maximum falls in December–February.

In [ ]:
with Scenario("historical") as hist:
    n = hist.available_steps()
    pts = [10_000, 50_000, 100_000]
    hs = hist.read("hs", pts, slice(0, n))

fig, ax = plt.subplots(figsize=(10, 3))
for i, p in enumerate(pts):
    lat, lon = grid["point_lat"][p], (grid["point_lon"][p] + 180) % 360 - 180
    label = f"{abs(lat):.1f}°{'S' if lat < 0 else 'N'}, {abs(lon):.1f}°{'W' if lon < 0 else 'E'}"
    ax.plot(assumed_times(n), hs[:, i], lw=0.4, label=label, rasterized=True)
ax.set(ylabel="Hs [m]", title="Significant wave height, 3-hourly (time axis assumed to start 1 Jan 1985)")
ax.legend()
save_fig(fig, "hs_timeseries_3points", FIG_DIR)
plt.show()

In [ ]:
# Seasonal cycle of Hs along 45°N (all sea points on that latitude row)
band = np.flatnonzero(np.isclose(grid["point_lat"], 45.0))
b0, b1 = band.min(), band.max() + 1          # points are stored latitude by latitude, so this is one block

fig, ax = plt.subplots(figsize=(8, 3.5))
for s in SCENARIOS:
    with Scenario(s) as sc:
        n = sc.available_steps()
        x = sc.read("hs", slice(b0, b1), slice(0, n))[:, band - b0]
    series = np.nanmean(x, axis=1)                                   # band mean per time step
    month = assumed_times(n).astype("datetime64[M]").astype(int) % 12   # 0 = January, if the file starts 1 Jan
    clim = [np.nanmean(series[month == m]) if (month == m).any() else np.nan for m in range(12)]
    ax.plot(range(1, 13), clim, marker="o", label=s)
ax.set(xticks=range(1, 13), xticklabels=list("JFMAMJJASOND"), ylabel="mean Hs [m]",
       title=f"Seasonal cycle of Hs along 45°N ({band.size} sea points), assuming each file starts 1 January")
ax.legend()
save_fig(fig, "hs_seasonal_cycle_45N", FIG_DIR)
plt.show()

## 4. Grid and coordinates

All three files share one grid: the sea mask defines 142,868 sea points on a global 0.5° grid (720 × 361), and the data store one column per sea point. Land is not stored at all.

In [ ]:
inspect_mat(RAW / "SeaMask_WW3.mat")

In [ ]:
# Mean Hs over the first 31 days: a quick check that point numbers match locations
with Scenario("historical") as hist:
    hs_month = hist.read("hs", slice(None), slice(0, 248))      # all points, first 31 days (~140 MB)
fig, ax = plot_map(grid, np.nanmean(hs_month, axis=0), label="mean Hs [m]",
                   title="Mean Hs, first 31 days (historical)")
save_fig(fig, "hs_map_first_month", FIG_DIR)
plt.show()

Find the point closest to any location (longitude can be −180…180 or 0…360):

In [ ]:
p = nearest_point(grid, lat=45.0, lon=-30.0)   # mid North Atlantic

## 5. Ten-year statistics of Hs for all three scenarios

Following Francesco's suggestion: per sea point, the **mean**, **standard deviation** and **99th percentile** of Hs over a 10-year period, for each scenario.

- **Period:** the first 10 years of each file. If the time axis assumptions hold, that is 1985–1994 for historical; the SSP years follow once Francesco confirms their period. Change `PERIOD_YEARS` or `START_YEAR` to use another window.
- **Run time:** the first run reads ~17 GB per scenario and takes several minutes per scenario (progress is printed). Results are cached, so later runs are instant.

In [ ]:
PERIOD_YEARS = 10
START_YEAR = 0                                          # years after the start of each file
t0 = int(START_YEAR * 365.25 * STEPS_PER_DAY)
t1 = t0 + int(PERIOD_YEARS * 365.25 * STEPS_PER_DAY)    # 10 years = 29,220 steps
PERIOD = slice(t0, t1)

stats = {s: period_stats(s, "hs", PERIOD) for s in SCENARIOS}

In [ ]:
STATS = [("mean", "mean Hs [m]", "viridis"),
         ("std", "std of Hs [m]", "magma"),
         ("p99", "99th percentile of Hs [m]", "inferno")]

fig, axes = plt.subplots(3, 3, figsize=(19, 12), constrained_layout=True)
for j, (key, label, cmap) in enumerate(STATS):
    vmin, vmax = np.nanpercentile(np.concatenate([stats[s][key] for s in SCENARIOS]), [1, 99])
    for i, s in enumerate(SCENARIOS):           # same colour scale in each column, so panels compare directly
        plot_map(grid, stats[s][key], ax=axes[i, j], title=f"{s}: {label}", label=label,
                 cmap=cmap, vmin=vmin, vmax=vmax)
fig.suptitle(f"Hs statistics over {PERIOD_YEARS} years (first {PERIOD_YEARS} years of each file)", fontsize=14)
save_fig(fig, f"hs_{PERIOD_YEARS}yr_stats_maps", FIG_DIR)
plt.show()

In [ ]:
# Change relative to historical (red = higher in the scenario, blue = lower)
fig, axes = plt.subplots(2, 3, figsize=(19, 8), constrained_layout=True)
for j, (key, label, _) in enumerate(STATS):
    diffs = {s: stats[s][key] - stats["historical"][key] for s in ("ssp126", "ssp585")}
    lim = np.nanpercentile(np.abs(np.concatenate(list(diffs.values()))), 99)
    for i, (s, d) in enumerate(diffs.items()):
        plot_map(grid, d, ax=axes[i, j], title=f"{s} − historical: {label}",
                 label=f"change in {label}", diverging=True, vmax=lim)
fig.suptitle(f"Change in Hs statistics relative to historical ({PERIOD_YEARS}-year periods)", fontsize=14)
save_fig(fig, f"hs_{PERIOD_YEARS}yr_stats_change_maps", FIG_DIR)
plt.show()

In [ ]:
# Global summary, weighted by area (cos latitude)
summary = pd.DataFrame({s: {"mean Hs [m]": area_mean(grid, stats[s]["mean"]),
                            "std of Hs [m]": area_mean(grid, stats[s]["std"]),
                            "p99 of Hs [m]": area_mean(grid, stats[s]["p99"])}
                        for s in SCENARIOS}).T
summary_change = (summary - summary.loc["historical"]).drop(index="historical")
display(summary)
display(summary_change.add_prefix("change in "))
save_table(summary, f"hs_{PERIOD_YEARS}yr_global_summary", FIG_DIR)

**What to look for:** where the changes are largest (e.g. Southern Ocean, North Atlantic, Arctic), whether SSP5-8.5 changes more than SSP1-2.6, and whether the extremes (99th percentile) change differently from the mean. Note that the comparison uses the first 10 years of each file, so its meaning depends on the SSP period that Francesco confirms.

## 6. Where are values missing?

Land is not stored, so "missing" here means NaN values at sea points. They appear at high latitudes and most likely mark **sea ice**, where the wave model does not compute waves *(to confirm with Francesco)*. The maps show the share of time steps that are missing over the 10-year period.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5), constrained_layout=True)
for ax, s in zip(axes, SCENARIOS):
    plot_map(grid, 100 * stats[s]["nan_frac"], ax=ax, title=f"{s}: share of time steps missing",
             label="missing [%]", cmap="Blues", vmin=0, vmax=100)
save_fig(fig, f"hs_{PERIOD_YEARS}yr_missing_maps", FIG_DIR)
plt.show()

missing = pd.DataFrame({s: {"never missing [% of points]": 100 * np.mean(stats[s]["nan_frac"] == 0),
                            "sometimes missing [%]": 100 * np.mean((stats[s]["nan_frac"] > 0) & (stats[s]["nan_frac"] < 1)),
                            "always missing [%]": 100 * np.mean(stats[s]["nan_frac"] == 1),
                            "missing, area-weighted [%]": 100 * area_mean(grid, stats[s]["nan_frac"])}
                        for s in SCENARIOS}).T
display(missing)
save_table(missing, f"hs_{PERIOD_YEARS}yr_missing_summary", FIG_DIR)

## 7. Do the datasets use comparable grids and variable definitions?

In [ ]:
rows = []
for s in SCENARIOS:
    with Scenario(s) as sc:
        rows.append({"scenario": s, "time steps": sc.shape[0], "sea points": sc.shape[1],
                     "chunks (time, points)": sc.chunks,
                     "data": "complete" if sc.complete else "partial"})
dims = pd.DataFrame(rows).set_index("scenario")
display(dims)
for col in ("time steps", "sea points", "chunks (time, points)"):
    print(f"{col:24s} identical in all files: {dims[col].astype(str).nunique() == 1}")
display(values["median"].unstack("scenario"))   # same variable, similar medians -> same definitions

## 8. Summary of findings

*Update after running the notebook; items marked "to confirm" are for Francesco.*

1. **Model and scenarios:** EC-Earth3 (CMIP6) wave and wind fields, most likely computed with WAVEWATCH III *(to confirm)*; one file each for historical, SSP1-2.6 and SSP5-8.5.
2. **Variables, units, dimensions:** `hs` [m], `fp` [Hz], `uwnd` and `vwnd` [m/s] *(units inferred, to confirm)*; each 87,656 time steps × 142,868 sea points, float32.
3. **Years and time step:** 30 years of 3-hourly data per file; historical most likely 1985–2014; SSP period *to confirm*. Seasonal-cycle check: *(see figure)*.
4. **Missing values:** land is not stored; NaNs at high latitudes, most likely sea ice *(see maps and table)*.
5. **Comparability:** same dimensions, chunking, sea mask and variable order in all files, and similar value ranges *(see section 7)*, so the datasets are directly comparable; the time periods still need confirming.